# Conferência das variáveis do conjunto curto no GEE

Coleção `projects/fcoliveira/assets/Climas2/xavier_conjunto_curto_decenal`: 9 variáveis climáticas em média móvel
de 10 anos (Xavier BR-DWGD v3.2.4), uma imagem por ano de 1985 a 2026, para o modelo de SOC do MapBiomas Solo.

O que este notebook confere:

1. **Mapa interativo** com seletor de variável e ano, *inspector* (clique mostra os valores), zoom e mapa de fundo;
   modos de visualização: valor (pixel ou bilinear), diferença para o ano anterior e bordas preenchidas.
2. **Série temporal** do ponto clicado (1985-2026).
3. **Cobertura:** nenhum pixel sem valor no Brasil, nas bordas, no litoral e nas ilhas, amostrando a 30 m com a
   interpolação bilinear (como o modelo vai usar).
4. **Faixas de valores por ano** e saltos entre anos consecutivos (para achar anos ou regiões estranhas).

Requer: `earthengine-api`, `geemap`, `ipywidgets`, `pandas`, `matplotlib`; acesso ao projeto `mapbiomas-brazil`.

In [ ]:
import ee
import geemap
import ipywidgets as w
import matplotlib.pyplot as plt
import pandas as pd

ee.Initialize(project='mapbiomas-brazil')

COLECAO = 'projects/fcoliveira/assets/Climas2/xavier_conjunto_curto_decenal'
col = ee.ImageCollection(COLECAO)
ANOS = list(range(1985, 2027))
VIS = {  # variável: (rótulo, mínimo, máximo, paleta)
    'cd_t_media': ('Temperatura média (°C)', 14, 28, 'RdYlBu_r'),
    'cd_tmax_media': ('Temperatura máxima média (°C)', 20, 35, 'RdYlBu_r'),
    'cd_tmin_media': ('Temperatura mínima média (°C)', 8, 24, 'RdYlBu_r'),
    'cd_t_mes_quente': ('T do mês mais quente (°C)', 18, 31, 'RdYlBu_r'),
    'cd_t_mes_frio': ('T do mês mais frio (°C)', 8, 28, 'RdYlBu_r'),
    'cd_amplitude': ('Amplitude térmica diária (°C)', 6, 16, 'magma'),
    'cd_biotemp': ('Biotemperatura (°C)', 12, 26, 'RdYlBu_r'),
    'cd_cdd': ('Dias secos consecutivos (dias)', 10, 150, 'YlOrBr'),
    'cd_arm_csolo': ('Armazenamento de água no solo, ARM (mm)', 0, 200, 'BrBG'),
}
BRASIL = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Brazil'))
print('imagens:', col.size().getInfo(), '| bandas:', col.first().bandNames().getInfo())

## 1. Mapa interativo

- **Variável, ano e modo** nos controles acima do mapa; a camada é trocada na hora.
- **Inspector:** ative o ícone de *inspector* na barra de ferramentas do mapa (ou use o clique, que também mostra a
  série do ponto na seção 2) e clique no mapa: aparecem os valores de todas as camadas.
- **Modos:** `valor (pixel)` mostra a grade de 0,1° como está no asset; `valor (bilinear)` mostra como o modelo vai
  ver em 30 m; `diferença p/ ano anterior` realça saltos; `bordas preenchidas` mostra onde o valor veio do vizinho
  mais próximo (km).

In [ ]:
def paleta(nome):
    from geemap import colormaps
    return colormaps.get_palette(nome, n_class=9)

m = geemap.Map(center=[-14, -52], zoom=4, height='650px')
m.add_basemap('SATELLITE')
m.add_basemap('ROADMAP')
try:
    m.add('inspector')
except Exception:
    pass
m.add_layer(ee.Image().byte().paint(BRASIL, 1, 1), {'palette': ['ffffff']}, 'limite do Brasil')

var = w.Dropdown(options=[(v[0], k) for k, v in VIS.items()], value='cd_t_media', description='variável')
ano = w.IntSlider(value=2020, min=ANOS[0], max=ANOS[-1], description='ano', continuous_update=False)
modo = w.Dropdown(options=['valor (pixel)', 'valor (bilinear)', 'diferença p/ ano anterior', 'bordas preenchidas'],
                  value='valor (bilinear)', description='modo')
estado = {'camada': None, 'barra': None}


def imagem_do_ano(a):
    return col.filter(ee.Filter.eq('year', a)).first()


def atualizar(*_):
    v, a, mo = var.value, ano.value, modo.value
    rot, vmin, vmax, pal = VIS[v]
    img = imagem_do_ano(a)
    if mo == 'valor (pixel)':
        camada, vis, titulo = img.select(v), {'min': vmin, 'max': vmax, 'palette': paleta(pal)}, f'{rot} {a} (pixel)'
    elif mo == 'valor (bilinear)':
        camada = img.select(v).resample('bilinear')
        vis, titulo = {'min': vmin, 'max': vmax, 'palette': paleta(pal)}, f'{rot} {a} (bilinear)'
    elif mo == 'diferença p/ ano anterior':
        if a == ANOS[0]:
            a = ANOS[1]; ano.value = a
        camada = img.select(v).subtract(imagem_do_ano(a - 1).select(v))
        lim = (vmax - vmin) / 20
        vis, titulo = {'min': -lim, 'max': lim, 'palette': paleta('RdBu_r')}, f'{rot}: {a} - {a - 1}'
    else:
        camada = img.select('dist_preenchimento_km').selfMask()
        vis, titulo = {'min': 0, 'max': 50, 'palette': ['fee0d2', 'de2d26']}, 'bordas preenchidas (km)'
    if estado['camada'] is not None:
        m.remove_layer(estado['camada'])
    if estado['barra'] is not None:
        m.remove(estado['barra'])
    m.add_layer(camada, vis, titulo)
    estado['camada'] = m.find_layer(titulo)
    m.add_colorbar(vis, label=titulo, layer_name=titulo)
    estado['barra'] = m.controls[-1]


for wd in (var, ano, modo):
    wd.observe(atualizar, names='value')
atualizar()
display(w.HBox([var, ano, modo]), m)

## 2. Série temporal do ponto clicado

Clique no mapa acima: o gráfico abaixo mostra a variável selecionada de 1985 a 2026 no ponto (valor do pixel e
bilinear). Uma série suave, sem degraus de um ano para o outro, é o esperado: cada ano é a média dos 10 anteriores.

In [ ]:
saida = w.Output()


def serie(lon, lat):
    p = ee.Geometry.Point([lon, lat])
    v = var.value

    def valores(img):
        a = img.select(v).reduceRegion(ee.Reducer.first(), p, 30)
        b = img.select(v).resample('bilinear').reduceRegion(ee.Reducer.first(), p, 30)
        return ee.Feature(None, {'ano': img.get('year'), 'pixel': a.get(v), 'bilinear': b.get(v)})
    df = pd.DataFrame([f['properties'] for f in ee.FeatureCollection(col.map(valores)).getInfo()['features']])
    return df.sort_values('ano')


def ao_clicar(**kw):
    if kw.get('type') != 'click':
        return
    lat, lon = kw['coordinates']
    with saida:
        saida.clear_output()
        df = serie(lon, lat)
        fig, ax = plt.subplots(figsize=(9, 3.2))
        ax.plot(df.ano, df.pixel, '-', color='#8a8984', lw=1.4, label='pixel 0,1°')
        ax.plot(df.ano, df.bilinear, '-', color='#2a78d6', lw=2, label='bilinear (como o modelo usa)')
        ax.set(title=f'{VIS[var.value][0]} em ({lat:.3f}, {lon:.3f})', xlabel='ano')
        ax.grid(alpha=.3); ax.legend()
        plt.show()
        print(df.round(2).to_string(index=False))


m.on_interaction(ao_clicar)
saida

## 3. Cobertura: nenhum pixel sem valor

Amostra a 30 m, com a interpolação bilinear, em (a) pontos críticos de borda, litoral e ilhas e (b) 5.000 pontos
aleatórios no Brasil, para todos os anos e variáveis. O esperado é **zero** valores ausentes.

In [ ]:
CRITICOS = {
    'Fernando de Noronha': (-32.42, -3.85), 'Trindade': (-29.32, -20.51), 'Atol das Rocas': (-33.80, -3.86),
    'Ponta do Seixas (extremo leste)': (-34.79, -7.15), 'Serra do Divisor (extremo oeste, AC)': (-73.98, -7.53),
    'Chuí (extremo sul)': (-53.37, -33.74), 'Monte Caburaí (extremo norte)': (-60.21, 5.27),
    'Oiapoque (foz)': (-51.63, 4.30), 'Ilha de Marajó (litoral)': (-49.45, -0.92), 'Ilhabela': (-45.36, -23.81),
    'Florianópolis (ilha)': (-48.55, -27.60), 'Pantanal (fronteira com a Bolívia)': (-57.70, -19.00),
    'Itatiaia (2.400 m)': (-44.67, -22.40), 'Foz do Iguaçu (tríplice fronteira)': (-54.58, -25.55),
}
pontos = ee.FeatureCollection([ee.Feature(ee.Geometry.Point(xy), {'local': k}) for k, xy in CRITICOS.items()])
aleatorios = ee.FeatureCollection.randomPoints(BRASIL.geometry(), 5000, 42)
VARS = list(VIS)


def amostrar(fc, anos):
    linhas = []
    for a in anos:
        img = imagem_do_ano(a).select(VARS).resample('bilinear')
        r = img.reduceRegions(fc, ee.Reducer.first().forEachBand(img), 30).getInfo()['features']
        for f in r:
            linhas.append({'ano': a, **f['properties']})
    return pd.DataFrame(linhas)


crit = amostrar(pontos, ANOS)
faltas = crit[VARS].isna().groupby(crit['local']).sum().sum(axis=1)
print('valores ausentes por local crítico (todos os anos e variáveis):')
print(faltas.to_string())
print('\nvalores em 2020:')
print(crit[crit.ano == 2020].set_index('local')[VARS].round(1).to_string())

In [ ]:
alea = amostrar(aleatorios, [1985, 2000, 2010, 2020, 2026])
print('pontos aleatórios:', alea.groupby('ano').size().to_dict())
print('valores ausentes por variável:', alea[VARS].isna().sum().to_dict())
print(alea[VARS].describe().round(2).to_string())

## 4. Faixas de valores por ano e saltos

Mínimo, média e máximo no Brasil em cada ano (grade de 0,1°). Linhas suaves são o esperado; um salto de um ano
para o outro, ou um valor fora da faixa física (por exemplo, ARM negativo, CDD acima de 365), indica problema.

In [ ]:
def resumo(img):
    r = img.select(VARS).reduceRegion(ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
                                      BRASIL.geometry(), 11132, maxPixels=1e9)
    return ee.Feature(None, r).set('ano', img.get('year'))


res = pd.DataFrame([f['properties'] for f in ee.FeatureCollection(col.map(resumo)).getInfo()['features']])
res = res.sort_values('ano').set_index('ano')
fig, axs = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for ax, v in zip(axs.flat, VARS):
    ax.fill_between(res.index, res[f'{v}_min'], res[f'{v}_max'], color='#cde2fb', label='mín-máx')
    ax.plot(res.index, res[f'{v}_mean'], color='#2a78d6', lw=2, label='média')
    ax.set_title(VIS[v][0], fontsize=10); ax.grid(alpha=.3)
axs.flat[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

saltos = res[[f'{v}_mean' for v in VARS]].diff().abs()
print('maior variação da média entre anos consecutivos:')
print(pd.DataFrame({'variação': saltos.max().round(3), 'ano': saltos.idxmax()}).to_string())
print('\nfaixas físicas (todos os anos):')
print(pd.DataFrame({'mín': res[[f'{v}_min' for v in VARS]].min().values,
                    'máx': res[[f'{v}_max' for v in VARS]].max().values}, index=VARS).round(2).to_string())